# 1 Loading

This notebook is meant to load all datasets necessary for the pipeline

## 1.1 Imports

In [2]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm

import utilities as u

pd.options.display.max_rows = 1000
print(Path('..').absolute())

c:\Users\User\Askar\Thesis\..


## 1.2 Check the initial file structure

In [4]:
u.check_file_structure(Path('.'),
                       {
                        "all_scpdb_entries.txt": None,
                        "excluded_scpdb_entries.txt": None,
                         "01_loading.ipynb": None,
                         "02_data.ipynb": None,
                        });

File Structure Check Report:
--------------------------
The expected structure:

├── all_scpdb_entries.txt
├── excluded_scpdb_entries.txt
├── 01_loading.ipynb
└── 02_data.ipynb

Missing directories:
	None

Extra directories:
	SIFTS
	Images
	Submission
	Logs
	for_bioinf
	Metrics
	SIFTS1
	GNNLogs
	CV
	Files

Missing files:
	None


## 1.3 Gather scPDB database

The cell below will download the current version of scPDB and store it in 

    Files/scPDB

Attention: the total size of the downloadable files might be 16 Gb + and the time required is several hours.

In [ ]:
all_entries = pd.read_csv('all_scpdb_entries.txt', header=None)
all_entries = sorted(all_entries.iloc[:, 0].to_list())

not_loaded = [entry for entry in all_entries
              if not all(file in os.listdir(os.path.join('Files/scPDB', entry))
                         for file in ('protein.mol2', 'site.mol2', 'ligand.mol2'))]

if not_loaded:
    u.download_scPDB(
        list_of_entries=not_loaded,
        files_to_load = {'protein.mol2', 'site.mol2', 'cavity.mol2', 'IPF.txt', 'interaction.mol2', 'ligand.mol2'},
        skip_existing_files = True,
        dir = 'Files/scPDB'
        );

## 1.4 Gather SCOPe

The cell below will download the current version of SCOPe and store it in 

    Files/SCOPe/[version].csv

In [3]:
u.download_SCOPe(
    version = '2.08',
    exist_ok = True,
    dir = 'Files/SCOPe'
)

Version: 2.08
Path: Files/SCOPe\2_08.csv
Version is already in the directory (10.4 Mb)


## 1.5 scPDB WebPages

There is a predefined list of 'excluded_entries', which I have already checked, and they are not fetchable.

The pages will be stored as Files/scPDB/entry/html.txt

The parsed page will be stored as Files/scPDB/entry/html.json

In [6]:
excluded_entries = open('excluded_scpdb_entries.txt', 'r').read().split('\n')

u.download_scPDB_web_pages(
    exist_ok=True,
    dir='Files/scPDB',
    exclude_entries=excluded_entries
)

u.parse_scPDB_pages(
    dir = 'Files/scPDB',
    exist_ok=True,
    scope_path = 'Files/SCOPe/2_08.csv'
)

Entries provided (from Files/scPDB): 16036
Expected size of the downloads: 611.7 Mb


Loading scPDB web pages...: 100%|██████████| 16036/16036 [00:01<00:00, 14660.11it/s]


Failed entries: 3
Parsing scPDB web pages
Entries provided (from Files/scPDB): 17597


Parsing Source Pages...: 100%|██████████| 17597/17597 [00:02<00:00, 8377.78it/s]

Entries found in the dir: 17594
Htmls found: 16033 (91.1 %)
Skipped parsings: 16033 (91.1 %)
Successfully parsed: 0 (0.0 %)
html.json now available for 16033 entries (91.1 %)
Errors occured: 0


## 1.6 RCSB

Using RCSB API, we can extract a lot of metadata from PDB instances (both about Protein and Ligand)

Each RCSB PDB entry has a complex structure:

- entry: general information about the resolved PDB structure
    - identifiers: mapping to subfiles
    - assemblies: assemblies reolved
    - entities: molecular entities (polymers / ligands / polysaccharides)
        - chains (chains of polymers)

All these entries are stores as .json files in the Files/scPDB/entry/

Note: several Gb can be loaded

In [7]:
u.fetch_RCSB_metadata(
    exist_ok = True,
    dir = 'Files/scPDB',
    exclude_entries = None
)

Fetching RCSB json files
Entries provided (from Files/scPDB): 17597
Expected size of the downloads: 1.1 Gb


Fetching RCSB jsons (127484 files, 1.5 Gb)...: 100%|██████████| 17597/17597 [50:12<00:00,  5.84it/s]   

Fetching finished (files fetched: 127484, actual size: 1.5 Gb; skipped as existing: 150901)
Failed entries: 2058


## 1.7 Load SIFTS Files

In [4]:
u.download_SIFTS(exist_ok=True,
                 dir='SIFTS')

Path: SIFTS
pdb_chain_enzyme.csv already exists - skipped
pdb_chain_uniprot.csv already exists - skipped
scop.csv already exists - skipped
scop_names.csv already exists - skipped
Files saved to the directory (0 b)


## 1.8 UniProt

Now we can fetch the UniProt metadata

The information will be stored as Files/scPDB/entry/[UniProt].json

In [1]:
import utilities as u

u.fetch_uniprot_metadata(
    exist_ok = True,
    dir = 'Files/scPDB',
    sifts_dir = 'SIFTS',
    exclude_entries = None
)

Fetching UniProt json files
Entries provided (from Files/scPDB): 17594
Expected size of the downloads: 1.6 Gb


Fetching UniProts...: 100%|██████████| 17594/17594 [00:29<00:00, 602.89it/s]
Fetching UniProt JSONs (All: 19797 files, 1.6 Gb, Loaded:  0 files, 0 b)...: 100%|██████████| 17594/17594 [00:29<00:00, 588.95it/s]   

Fetching finished (files fetched: 19798, actual size: 1.6 Gb; skipped as existing: 19798)
Failed entries: 109


At this point, the folder of one entry should look like:

In [6]:
example = '1a2b_1'
example_path = os.path.join('Files', 'scPDB', example)
files = os.listdir(example_path)
print(example + '\\')
for file in sorted(files, key=lambda x: x.split('.')[1] + x.split('.')[0]):
    if file.split('.')[-1] in ['json', 'mol2', 'txt']:
        print('\t'  + file)

1a2b_1\
	P61586.json
	assembly_1.json
	compiled.json
	html.json
	identifiers.json
	rcsb_assembly_1.json
	rcsb_entity_1.json
	rcsb_entity_1_chain_A.json
	rcsb_entity_2.json
	rcsb_entity_3.json
	rcsb_entry.json
	rcsb_identifiers.json
	cavity.mol2
	interaction.mol2
	ligand.mol2
	protein.mol2
	site.mol2
	FASTA.txt
	FASTAsite.txt
	IPF.txt
	html.txt
